In [ ]:
import math
import pandas as pd

# 1. DATASET: Loan Approval Data
data = {
    "Income": ["High", "High", "Low", "Medium", "Medium", "Low", "High"],
    "Credit": ["Good", "Bad", "Good", "Good", "Bad", "Bad", "Good"],
    "Employed": ["Yes", "Yes", "No", "Yes", "Yes", "No", "No"],
    "Approved": ["Yes", "Yes", "No", "Yes", "No", "No", "Yes"],  # Target
}
df = pd.DataFrame(data)

# 2. FOIL GAIN CALCULATION
def get_foil_gain(p0, n0, p1, n1):
  if p1 == 0:
    return -999
  # Info before adding condition vs info after
  info_before = math.log2(p0 / (p0 + n0))
  info_after = math.log2(p1 / (p1 + n1))
  return p1 * (info_after - info_before)


# 3. FOIL ALGORITHM
def run_foil(df, target_col, pos_label):
  positives = df[df[target_col] == pos_label]
  negatives = df[df[target_col] != pos_label]
  features = [c for c in df.columns if c != target_col]

  rules = []

  # OUTER LOOP: Sequential Covering (Run until all Positives are covered)
  while len(positives) > 0:
    rule = []
    curr_pos = positives.copy()
    curr_neg = negatives.copy()

    # INNER LOOP: Build one rule by adding conditions greedily
    while len(curr_neg) > 0:
      best_gain = -999
      best_condition = None

      # Try every feature-value pair (e.g., Credit == Good)
      for col in features:
        for val in df[col].unique():
          if (col, val) in rule:
            continue

          # Count matching positives and negatives
          p1 = len(curr_pos[curr_pos[col] == val])
          n1 = len(curr_neg[curr_neg[col] == val])

          # Calculate Gain
          gain = get_foil_gain(len(curr_pos), len(curr_neg), p1, n1)

          if gain > best_gain:
            best_gain = gain
            best_condition = (col, val)

      # If no condition helps, stop
      if best_gain <= 0 or best_condition is None:
        break

      # Add best condition to the rule
      rule.append(best_condition)
      curr_pos = curr_pos[curr_pos[best_condition[0]] == best_condition[1]]
      curr_neg = curr_neg[curr_neg[best_condition[0]] == best_condition[1]]

    if not rule:
      break

    # Save rule and REMOVE covered positive data (Separate step)
    rules.append(rule)
    positives = positives.drop(curr_pos.index)

  return rules


# 4. RUN AND PRINT
learned_rules = run_foil(df, target_col="Approved", pos_label="Yes")

print("--- LEARNED FOIL RULES ---")
for i, rule in enumerate(learned_rules, 1):
  cond_str = " AND ".join([f"{col} == '{val}'" for col, val in rule])
  print(f"Rule {i}: IF {cond_str} THEN Approved = Yes")

--- LEARNED FOIL RULES ---
Rule 1: IF Income == 'High' THEN Approved = Yes
Rule 2: IF Income == 'Medium' AND Credit == 'Good' THEN Approved = Yes


In [ ]:
import math
import pandas as pd

# --------------------------------------------------
# 1. DATASET
# --------------------------------------------------

data = {
    "Income":   ["High", "High", "Low", "Medium", "Medium", "Low", "High"],
    "Credit":   ["Good", "Bad", "Good", "Good", "Bad", "Bad", "Good"],
    "Employed": ["Yes", "Yes", "No", "Yes", "Yes", "No", "No"],
    "Approved": ["Yes", "Yes", "Yes", "Yes", "No", "No", "Yes"]
}

df = pd.DataFrame(data)


# --------------------------------------------------
# 2. FOIL GAIN
# --------------------------------------------------

def foil_gain(p0, n0, p1, n1):

    if p1 == 0:
        return -999

    before = math.log2(p0 / (p0 + n0))
    after = math.log2(p1 / (p1 + n1))

    return p1 * (after - before)


# --------------------------------------------------
# 3. FOIL ALGORITHM
# --------------------------------------------------

def foil(df):

    positives = df[df["Approved"] == "Yes"]
    negatives = df[df["Approved"] == "No"]

    features = ["Income", "Credit", "Employed"]

    rules = []

    # Repeat until all positive examples are covered
    while len(positives) > 0:

        rule = []
        current_pos = positives.copy()
        current_neg = negatives.copy()

        # Build one rule
        while len(current_neg) > 0:

            best_gain = -999
            best_condition = None

            # Try every possible condition
            for col in features:

                for val in df[col].unique():

                    p0 = len(current_pos)
                    n0 = len(current_neg)

                    p1 = len(current_pos[current_pos[col] == val])
                    n1 = len(current_neg[current_neg[col] == val])

                    gain = foil_gain(p0, n0, p1, n1)

                    if gain > best_gain:
                        best_gain = gain
                        best_condition = (col, val)

            # Stop if no useful condition exists
            if best_gain <= 0:
                break

            # Add best condition
            rule.append(best_condition)

            col, val = best_condition

            current_pos = current_pos[current_pos[col] == val]
            current_neg = current_neg[current_neg[col] == val]

        # Save the rule
        if not rule:
            break

        rules.append(rule)

        # Remove positive examples covered by this rule
        positives = positives.drop(current_pos.index)

    return rules


# --------------------------------------------------
# 4. RUN FOIL
# --------------------------------------------------

rules = foil(df)


# --------------------------------------------------
# 5. DISPLAY RULES
# --------------------------------------------------

print("\n===== FINAL FOIL RULES =====")

for i, rule in enumerate(rules, 1):

    conditions = ""

    for col, val in rule:

        condition = f"{col} = '{val}'"

        if conditions == "":
            conditions = condition
        else:
            conditions = conditions + " AND " + condition

    print(f"Rule {i}: IF {conditions} THEN Approved = Yes")


===== FINAL FOIL RULES =====
Rule 1: IF Credit = 'Good' THEN Approved = Yes
Rule 2: IF Income = 'High' THEN Approved = Yes
